
# Homework 5 — PDE Pricing

## Pricing the Maximum of Two Independent Stocks

We consider two independent stock processes with equal volatility and initial prices:

$$
S_1(0)=S_2(0)=100
$$

At maturity, the contract pays the value of the more expensive stock:

$$
V(T)=\max(S_1(T),S_2(T))
$$

## Part (a): Analytical Fair Value

We derive the fair value of the contract under the risk-neutral measure.

## Part (b): Partial Differential Equation

We derive the two-dimensional Black-Scholes PDE and its terminal condition.

## Part (c): Finite Difference Method

We solve the PDE numerically using a finite difference method.

## Part (d): Monte Carlo Comparison

We compare the PDE price with the Monte Carlo estimate and analyze the numerical difference.



### Part (a) — Fair Value

We have two independent stocks, and both start at 100.

At maturity, we receive the price of the more expensive stock:

$$
V(T) = \max(S_1(T), S_2(T))
$$

I assume that both stocks follow the Black-Scholes model,
with the same volatility and no dividends.

We can write the payoff as:

$$
\max(S_1(T), S_2(T))
= S_2(T) + \max(S_1(T)-S_2(T),0)
$$

This means we can use the exchange option formula.

Because the stocks are independent and have the same volatility:

$$
\sigma_{ex} = \sqrt{2}\sigma
$$

The fair value formula is:

$$
V(0) = S_1(0)N(d_1) + S_2(0)N(-d_2)
$$

where:

$$
d_1 = \frac{\ln(S_1(0)/S_2(0))+\sigma_{ex}^2T/2}
{\sigma_{ex}\sqrt{T}}
$$

$$
d_2 = d_1-\sigma_{ex}\sqrt{T}
$$

Since both starting prices are 100, we get:

$$
V(0) = 200N\left(\sigma\sqrt{T/2}\right)
$$

I will use this formula to calculate the price in Python.


In [1]:

import numpy as np
from scipy.stats import norm

# Starting values
S1 = 100
S2 = 100

# I use these values as an example
sigma = 0.20
T = 1
r = 0.05

# Both stocks have the same volatility
sigma_ex = np.sqrt(2) * sigma

# Calculate d1 and d2
d1 = (np.log(S1 / S2) + 0.5 * sigma_ex**2 * T) / (sigma_ex * np.sqrt(T))
d2 = d1 - sigma_ex * np.sqrt(T)

# Calculate the fair value
price = S1 * norm.cdf(d1) + S2 * norm.cdf(-d2)

print("Fair value =", round(price, 4))


Fair value = 111.2463



## Part (b) — Derivation of the PDE

We assume that the two stock prices follow the Black-Scholes model.

Since the stocks are independent, their Brownian motions are independent.

Under the risk-neutral measure:

$$
dS_1 = rS_1dt + \sigma S_1dW_1
$$

$$
dS_2 = rS_2dt + \sigma S_2dW_2
$$

Let $V(t,S_1,S_2)$ be the value of the contract at time $t$.

Using Ito's lemma, we get:

$$
dV =
\left(
\frac{\partial V}{\partial t}
+ rS_1\frac{\partial V}{\partial S_1}
+ rS_2\frac{\partial V}{\partial S_2}
+ \frac{\sigma^2 S_1^2}{2}\frac{\partial^2 V}{\partial S_1^2}
+ \frac{\sigma^2 S_2^2}{2}\frac{\partial^2 V}{\partial S_2^2}
\right)dt
+ \sigma S_1\frac{\partial V}{\partial S_1}dW_1
+ \sigma S_2\frac{\partial V}{\partial S_2}dW_2
$$

There is no mixed second derivative term because the two stocks are independent.

Using the risk-neutral pricing condition, the PDE becomes:

$$
\boxed{
\frac{\partial V}{\partial t}
+ \frac{\sigma^2 S_1^2}{2}\frac{\partial^2 V}{\partial S_1^2}
+ \frac{\sigma^2 S_2^2}{2}\frac{\partial^2 V}{\partial S_2^2}
+ rS_1\frac{\partial V}{\partial S_1}
+ rS_2\frac{\partial V}{\partial S_2}
- rV = 0
}
$$

The final condition at maturity is:

$$
V(T,S_1,S_2)=\max(S_1,S_2)
$$

This is a two-dimensional Black-Scholes PDE because the contract depends on two stock prices.

In the next part, I will solve this PDE numerically using the finite difference method.



## Part (c) — Finite Difference Method

I use the explicit finite difference method to solve the PDE numerically.

First, I create a grid for the two stock prices, S1 and S2.

At maturity, the value is:

$$
V(T,S_1,S_2)=\max(S_1,S_2)
$$

I start from the payoff at maturity and calculate the values backward in time.

For the first and second derivatives, I use central finite differences.

I use these parameters:

- Initial stock prices: 100 and 100
- Volatility: 20%
- Risk-free rate: 5%
- Maturity: 1 year
- Maximum stock price on the grid: 300

At the lower boundaries, the value equals the remaining stock price. At the upper boundaries, I approximate the value by 300.

Finally, I compare the numerical result with the analytical price from Part (a).


In [2]:

# Part (c) - Explicit finite difference method

import numpy as np

# Same parameters as Part (a)
S1 = 100
S2 = 100
sigma = 0.20
r = 0.05
T = 1.0

# Grid settings
Smax = 300
N = 101
time_steps = 1200

stock = np.linspace(0, Smax, N)
dS = stock[1] - stock[0]
dt = T / time_steps

# Two-dimensional stock price grid
X, Y = np.meshgrid(stock, stock, indexing="ij")

# Payoff at maturity
V = np.maximum(X, Y)

# Solve the PDE backward from maturity
for step in range(time_steps):

    middle = V[1:-1, 1:-1]

    # Second derivatives
    V_s1s1 = (
        V[2:, 1:-1] - 2 * middle + V[:-2, 1:-1]
    ) / dS**2

    V_s2s2 = (
        V[1:-1, 2:] - 2 * middle + V[1:-1, :-2]
    ) / dS**2

    # First derivatives
    V_s1 = (
        V[2:, 1:-1] - V[:-2, 1:-1]
    ) / (2 * dS)

    V_s2 = (
        V[1:-1, 2:] - V[1:-1, :-2]
    ) / (2 * dS)

    s1 = X[1:-1, 1:-1]
    s2 = Y[1:-1, 1:-1]

    # Black-Scholes PDE
    change = (
        0.5 * sigma**2 * s1**2 * V_s1s1
        + 0.5 * sigma**2 * s2**2 * V_s2s2
        + r * s1 * V_s1
        + r * s2 * V_s2
        - r * middle
    )

    V_new = V.copy()

    V_new[1:-1, 1:-1] = middle + dt * change

    # Boundary conditions
    V_new[0, :] = stock
    V_new[:, 0] = stock
    V_new[-1, :] = Smax
    V_new[:, -1] = Smax

    V = V_new

# Interpolate the price at S1 = 100, S2 = 100
i = int(S1 / dS)
j = int(S2 / dS)

a = (S1 - stock[i]) / dS
b = (S2 - stock[j]) / dS

fdm_price = (
    (1-a) * (1-b) * V[i, j]
    + a * (1-b) * V[i+1, j]
    + (1-a) * b * V[i, j+1]
    + a * b * V[i+1, j+1]
)

# Compare with Part (a)
analytical_price = 200 * norm.cdf(sigma * np.sqrt(T / 2))

print("FDM price =", round(fdm_price, 4))
print("Analytical price =", round(analytical_price, 4))
print("Absolute error =", round(abs(fdm_price - analytical_price), 4))


FDM price = 111.2591
Analytical price = 111.2463
Absolute error = 0.0128



## Part (d) — Monte Carlo Method

In this part, I use Monte Carlo simulation to estimate the price of the contract.

The two stocks are independent and follow the Black-Scholes model.

I simulate their prices at maturity using random numbers.

For each simulation, I calculate the payoff:

$$
V(T)=\max(S_1(T),S_2(T))
$$

Then I discount the average payoff to get the price today:

$$
V(0)\approx e^{-rT}\frac{1}{N}
\sum_{i=1}^{N}\max(S_1^{(i)}(T),S_2^{(i)}(T))
$$

Finally, I compare the Monte Carlo price with the analytical price and the Finite Difference Method price.


In [3]:

# Part (d) - Monte Carlo simulation

import numpy as np

# Use the same parameters as before
n_simulations = 200000
rng = np.random.default_rng(42)

# Independent random numbers for both stocks
Z1 = rng.standard_normal(n_simulations)
Z2 = rng.standard_normal(n_simulations)

# Simulate stock prices at maturity
S1_T = S1 * np.exp(
    (r - 0.5 * sigma**2) * T
    + sigma * np.sqrt(T) * Z1
)

S2_T = S2 * np.exp(
    (r - 0.5 * sigma**2) * T
    + sigma * np.sqrt(T) * Z2
)

# Contract payoff
payoff = np.maximum(S1_T, S2_T)

# Discount payoff to today
discounted_payoff = np.exp(-r * T) * payoff

# Monte Carlo price
mc_price = np.mean(discounted_payoff)

# Monte Carlo standard error
mc_se = np.std(
    discounted_payoff, ddof=1
) / np.sqrt(n_simulations)

# Compare results
print("Analytical price =", round(analytical_price, 4))
print("FDM price        =", round(fdm_price, 4))
print("Monte Carlo price=", round(mc_price, 4))

print()
print("FDM error =", round(abs(fdm_price - analytical_price), 4))
print("MC error  =", round(abs(mc_price - analytical_price), 4))
print("MC standard error =", round(mc_se, 4))
print("MC 95% confidence interval =",
      (round(mc_price - 1.96 * mc_se, 4),
       round(mc_price + 1.96 * mc_se, 4)))


Analytical price = 111.2463
FDM price        = 111.2591
Monte Carlo price= 111.2815

FDM error = 0.0128
MC error  = 0.0352
MC standard error = 0.042
MC 95% confidence interval = (np.float64(111.1993), np.float64(111.3638))



## Results and Conclusion

In this homework, I calculated the price of a contract that pays the maximum of two independent stock prices at maturity.

I used three methods:

- Analytical formula
- Finite Difference Method (FDM)
- Monte Carlo simulation

### Results

| Method | Price |
|---|---:|
| Analytical | 111.2463 |
| FDM | 111.2591 |
| Monte Carlo | 111.2815 |

The FDM price is very close to the analytical price, with an absolute error of 0.0128.

The Monte Carlo price is also close, with an absolute error of 0.0352 and a standard error of 0.0420.

The 95% confidence interval for the Monte Carlo estimate is approximately [111.1993, 111.3638]. The analytical price is inside this interval.

### Conclusion

The three pricing methods give similar results.

In this example, the Finite Difference Method gives a smaller error than the Monte Carlo simulation.

The results show that both numerical methods can be used to approximate the fair value of this contract.

The numerical example uses a volatility of 20%, a risk-free rate of 5%, and a maturity of one year.


### Additional Check — Grid Convergence

I repeat the finite difference calculation with different grid sizes.

I want to check if increasing the number of grid points makes the numerical price closer to the analytical price.

In [4]:

# Check convergence of the finite difference method

def solve_fdm(N):
    Smax = 300
    stock = np.linspace(0, Smax, N)
    dS = stock[1] - stock[0]

    # Use more time steps for finer grids
    time_steps = max(600, int(0.12 * (N - 1)**2))
    dt = T / time_steps

    X, Y = np.meshgrid(stock, stock, indexing="ij")
    V = np.maximum(X, Y)

    s1 = X[1:-1, 1:-1]
    s2 = Y[1:-1, 1:-1]

    for _ in range(time_steps):
        middle = V[1:-1, 1:-1]

        d11 = (V[2:, 1:-1] - 2*middle + V[:-2, 1:-1]) / dS**2
        d22 = (V[1:-1, 2:] - 2*middle + V[1:-1, :-2]) / dS**2

        d1 = (V[2:, 1:-1] - V[:-2, 1:-1]) / (2*dS)
        d2 = (V[1:-1, 2:] - V[1:-1, :-2]) / (2*dS)

        change = (
            0.5 * sigma**2 * (s1**2 * d11 + s2**2 * d22)
            + r * (s1*d1 + s2*d2)
            - r * middle
        )

        V_new = V.copy()
        V_new[1:-1, 1:-1] = middle + dt * change

        V_new[0, :] = stock
        V_new[:, 0] = stock
        V_new[-1, :] = Smax
        V_new[:, -1] = Smax

        V = V_new

    # For these grids, 100 is exactly a grid point
    i = np.argmin(abs(stock - S1))
    j = np.argmin(abs(stock - S2))

    return V[i, j]


grid_sizes = [61, 91, 121, 151]

print("Grid size | FDM price | Absolute error")
print("-" * 42)

for N in grid_sizes:
    result = solve_fdm(N)
    error = abs(result - analytical_price)

    print(f"{N:9d} | {result:9.4f} | {error:.6f}")


Grid size | FDM price | Absolute error
------------------------------------------
       61 |  111.2035 | 0.042805
       91 |  111.2277 | 0.018552
      121 |  111.2359 | 0.010425
      151 |  111.2396 | 0.006668



### Grid Convergence Results

I tested the finite difference method with different grid sizes.

When I increased the number of grid points, the numerical price became closer to the analytical price.

For 61 grid points, the absolute error was 0.042805.

For 151 grid points, the error decreased to 0.006668.

This shows that the finite difference solution converges towards the analytical result as the grid becomes finer.

The numerical method gives a good approximation of the contract price.
